In [3]:
import keras
import pandas as pd
import numpy as np
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings("ignore")

from mlflow.models import infer_signature


In [4]:
## load the dataset
data=pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)

In [5]:
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [6]:
data.columns

Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality'],
      dtype='str')

In [7]:
train, test = train_test_split(data, test_size=0.2, random_state=42)

In [8]:
train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
4665,7.3,0.17,0.36,8.20,0.028,44.0,111.0,0.99272,3.14,0.41,12.4,6
1943,6.3,0.25,0.44,11.60,0.041,48.0,195.0,0.99680,3.18,0.52,9.5,5
3399,5.6,0.32,0.33,7.40,0.037,25.0,95.0,0.99268,3.25,0.49,11.1,6
843,6.9,0.19,0.35,1.70,0.036,33.0,101.0,0.99315,3.21,0.54,10.8,7
2580,7.7,0.30,0.26,18.95,0.053,36.0,174.0,0.99976,3.20,0.50,10.4,5
...,...,...,...,...,...,...,...,...,...,...,...,...
4426,6.2,0.21,0.52,6.50,0.047,28.0,123.0,0.99418,3.22,0.49,9.9,6
466,7.0,0.14,0.32,9.00,0.039,54.0,141.0,0.99560,3.22,0.43,9.4,6
3092,7.6,0.27,0.52,3.20,0.043,28.0,152.0,0.99129,3.02,0.53,11.4,6
3772,6.3,0.24,0.29,13.70,0.035,53.0,134.0,0.99567,3.17,0.38,10.6,6


In [23]:
train_x = train.drop("quality", axis=1).values
train_y = train["quality"].values.ravel()

#test dataset
test_x = test.drop("quality", axis=1).values
test_y = test["quality"].values.ravel()

##spilling the train data into train and validation data
train_x, valid_x, train_y, valid_y = train_test_split(train_x, train_y, test_size=0.2, random_state=42)


In [24]:
signature = infer_signature(train_x, train_y)


In [25]:
np.mean(train_x,axis=0)

array([6.87086790e+00, 2.77890874e-01, 3.33854499e-01, 6.36080089e+00,
       4.55105297e-02, 3.49082642e+01, 1.37556637e+02, 9.94029683e-01,
       3.19041800e+00, 4.89722399e-01, 1.05203340e+01])

In [26]:
train_x.shape[1]

11

In [ ]:
### ANN Model
import mlflow
def train_model(params,epochs,train_x,train_y,valid_x,valid_y,test_x,test_y):

    ## Define model architecture
    mean=np.mean(train_x,axis=0)
    var=np.var(train_x,axis=0)

    model=keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean,variance=var),
            keras.layers.Dense(64,activation='relu'),
            keras.layers.Dense(1)

        ]
    )

    ## compile the model
    model.compile(optimizer=keras.optimizers.SGD(
        learning_rate=params["lr"],momentum=params["momentum"]
    ),
    loss="mean_squared_error",
    metrics=[keras.metrics.RootMeanSquaredError()]
    )

    ## Train the ANN model with lr and momentum params wwith MLFLOW tracking
    with mlflow.start_run(nested=True):
        model.fit(train_x,train_y,validation_data=(valid_x,valid_y),
                  epochs=epochs,
                  batch_size=64)
        
        ## Evaluate the model
        eval_result=model.evaluate(valid_x,valid_y,batch_size=64)

        eval_rmse=eval_result[1]

        ## Log the parameters and results
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse",eval_rmse)

        ## log the model

        mlflow.tensorflow.log_model(model,"model",signature=signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

In [28]:
def objective(params):
    # MLflow will track the parameters and results for each run
    result = train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y,
    )
    return result

In [30]:
space={
    "lr":hp.loguniform("lr",np.log(1e-5),np.log(1e-1)),
    "momentum":hp.uniform("momentum",0.0,1.0)

}

In [31]:

import mlflow


mlflow.set_experiment("wine-quality")
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials=Trials()
    best=fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")


Epoch 1/3                                            

 1/49 ━━━━━━━━━━━━━━━━━━━━ 17s 360ms/step - loss: 30.4102 - root_mean_squared_error: 5.5145
44/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 2.4363 - root_mean_squared_error: 1.5609    
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 2.2427 - root_mean_squared_error: 1.4976 - val_loss: 0.5512 - val_root_mean_squared_error: 0.7424

Epoch 2/3                                            

 1/49 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.5926 - root_mean_squared_error: 0.7698
42/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5720 - root_mean_squared_error: 0.7563 
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5800 - root_mean_squared_error: 0.7616 - val_loss: 0.5154 - val_root_mean_squared_error: 0.7179

Epoch 3/3                                            

 1/49 ━━━━━━━━━━━━━━━━━━━━ 2s 56ms/step - loss: 0.7203 - root_mean_squared_error: 0.8487
35/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5617 - root_mean_squared_error: 0.7495 
49/

2026/10/10 19:18:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 22s 470ms/step - loss: 35.3605 - root_mean_squared_error: 5.9465
31/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.7460 - root_mean_squared_error: 2.1785    
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 3.4623 - root_mean_squared_error: 1.8607 - val_loss: 1.3856 - val_root_mean_squared_error: 1.1771

Epoch 2/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 1.1015 - root_mean_squared_error: 1.0495
42/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.0413 - root_mean_squared_error: 1.0205 
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.0007 - root_mean_squared_error: 1.0004 - val_loss: 1.0547 - val_root_mean_squared_error: 1.0270

Epoch 3/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.8598 - root_mean_squared_error: 0.9272
34/49 ━━━━━━━━

2026/10/10 19:18:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Epoch 1/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 14s 309ms/step - loss: 34.8226 - root_mean_squared_error: 5.9011
47/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 4.9515 - root_mean_squared_error: 2.2252    
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 4.8149 - root_mean_squared_error: 2.1943 - val_loss: 1.8097 - val_root_mean_squared_error: 1.3453

Epoch 2/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 1.3210 - root_mean_squared_error: 1.1493
27/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.3364 - root_mean_squared_error: 1.1560 
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.3324 - root_mean_squared_error: 1.1543 - val_loss: 1.3289 - val_root_mean_squared_error: 1.1528

Epoch 3/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 1.1149 - root_mean_squared_error: 1.0559
48/49 ━━━━━━━━

...

Epoch 1/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 20s 424ms/step - loss: 33.8477 - root_mean_squared_error: 5.8179
30/49 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 25.1844 - root_mean_squared_error: 5.0184   
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 21.4692 - root_mean_squared_error: 4.6335 - val_loss: 13.2041 - val_root_mean_squared_error: 3.6337

Epoch 2/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 12.7643 - root_mean_squared_error: 3.5727
39/49 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 9.3808 - root_mean_squared_error: 3.0628  
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 8.7272 - root_mean_squared_error: 2.9542 - val_loss: 5.9473 - val_root_mean_squared_error: 2.4387

Epoch 3/3                                                                      

 1/49 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 4.9635 - root_mean_squared_error: 2.2279
34/49 ━━━━

2026/10/10 19:19:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



100%|██████████| 4/4 [00:35<00:00,  8.83s/trial, best loss: 0.7030555009841919]

2026/10/10 19:19:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best parameters: {'lr': np.float64(0.09180202031945797), 'momentum': np.float64(0.601736302174733)}
Best eval rmse: 0.7030555009841919


In [ ]:
artifact_path: file:c:/Users/aksha/OneDrive/Documents/Desktop/dl_mlflow/mlruns/1/models/m-7bf10056e9cb482380cc3b6e0e23e8e1/artifacts
flavors:
  python_function:
    data: data
    env:
      conda: conda.yaml
      virtualenv: python_env.yaml
    loader_module: mlflow.tensorflow
    python_version: 3.13.5
  tensorflow:
    code: null
    data: data
    keras_version: 2.21.0
    model_type: keras
    save_format: tf
is_signature_from_type_hint: false
mlflow_version: 3.17.0
model_id: m-7bf10056e9cb482380cc3b6e0e23e8e1
model_size_bytes: 28945
model_uuid: m-7bf10056e9cb482380cc3b6e0e23e8e1
prompts: null
run_id: 05c4e15530dc478fba1263910a3e8040
signature:
  inputs: '[{"type": "tensor", "tensor-spec": {"dtype": "float64", "shape": [-1, 11]}}]'
  outputs: '[{"type": "tensor", "tensor-spec": {"dtype": "int64", "shape": [-1]}}]'
  params: null
type_hint_from_example: false
utc_time_created: '2026-10-10 13:49:12.453082'
